21.1 — Setup MIL Wav2Vec2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import normalize
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

PROJECT_DIR = Path(
    r"C:\Users\acer\Desktop\ProgettoTesi"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "risultati"
)

EMBEDDING_DIR = (
    RESULTS_DIR
    / "embedding_multimodali_turn_level"
)

W2V_DIR = (
    RESULTS_DIR
    / "embedding_wav2vec2_turn_level"
)

CLINICAL_DIR = (
    PROJECT_DIR
    / "dati clinici"
)

MIL_DIR = (
    RESULTS_DIR
    / "mil_wav2vec2_dn4"
)

MIL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Metadata dei 3710 turni
metadata_df = pd.read_csv(
    EMBEDDING_DIR
    / "metadata_turni_multimodali.csv"
)

# DN4
dn4_df = pd.read_csv(
    CLINICAL_DIR
    / "patient_id_dn4_score.csv"
)

dn4_patient = (
    dn4_df[
        [
            "patient_id",
            "dn4_score"
        ]
    ]
    .dropna()
    .copy()
)

dn4_patient[
    "patient_id"
] = (
    dn4_patient[
        "patient_id"
    ].astype(int)
)


dn4_patient = (
    dn4_patient
    .drop_duplicates(
        subset=["patient_id"]
    )
)

# Metadata + DN4 + indice embedding
metadata_indexed = (
    metadata_df
    .reset_index()
    .rename(
        columns={
            "index":
                "embedding_row"
        }
    )
)

turn_dn4 = (
    metadata_indexed
    .merge(
        dn4_patient,
        on="patient_id",
        how="inner",
        validate="many_to_one"
    )
    .sort_values(
        "embedding_row"
    )
    .reset_index(
        drop=True
    )
)

turn_dn4[
    "dn4_binary"
] = (
    turn_dn4[
        "dn4_score"
    ] >= 4
).astype(int)

# Wav2Vec2
X_wav2vec2 = np.load(
    W2V_DIR
    / "embedding_audio_wav2vec2_base_turn_level.npy"
)

X_wav2vec2 = normalize(
    X_wav2vec2,
    norm="l2"
).astype(
    np.float32
)

# Tabella patient-level
patient_table = (
    turn_dn4[
        [
            "patient_id",
            "dn4_binary"
        ]
    ]
    .drop_duplicates(
        subset=["patient_id"]
    )
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

patient_ids = (
    patient_table[
        "patient_id"
    ].to_numpy()
)


patient_labels = (
    patient_table[
        "dn4_binary"
    ].to_numpy()
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

patient_folds = []

for (
    train_patient_idx,
    test_patient_idx
) in skf.split(
    patient_ids,
    patient_labels
):

    train_patients = (
        patient_ids[
            train_patient_idx
        ]
    )

    test_patients = (
        patient_ids[
            test_patient_idx
        ]
    )

    patient_folds.append(
        (
            train_patients,
            test_patients
        )
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# QC
print(
    "Device:",
    DEVICE
)

print(
    "Turni:",
    len(turn_dn4)
)

print(
    "Pazienti:",
    len(patient_ids)
)

print(
    "Classe 0:",
    int(
        np.sum(
            patient_labels == 0
        )
    )
)

print(
    "Classe 1:",
    int(
        np.sum(
            patient_labels == 1
        )
    )
)

print(
    "Wav2Vec2:",
    X_wav2vec2.shape
)

print(
    "Fold:",
    len(patient_folds)
)

Device: cpu
Turni: 3710
Pazienti: 90
Classe 0: 44
Classe 1: 46
Wav2Vec2: (3710, 768)
Fold: 5


21.2 — Attention MIL

In [3]:
class AttentionMIL(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=64,
        attention_dim=32,
        dropout=0.20
    ):

        super().__init__()

        self.instance_encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.attention = nn.Sequential(
            nn.Linear(
                hidden_dim,
                attention_dim
            ),
            nn.Tanh(),
            nn.Linear(
                attention_dim,
                1
            )
        )

        self.classifier = nn.Linear(
            hidden_dim,
            1
        )


    def forward(
        self,
        bag
    ):

        # bag:
        # [n_turns, input_dim]

        h = self.instance_encoder(
            bag
        )

        attention_logits = (
            self.attention(h)
            .squeeze(-1)
        )

        attention_weights = (
            torch.softmax(
                attention_logits,
                dim=0
            )
        )

        bag_embedding = torch.sum(
            attention_weights.unsqueeze(-1)
            * h,
            dim=0
        )

        logit = self.classifier(
            bag_embedding
        ).squeeze()

        return (
            logit,
            attention_weights
        )

21.3 — Helper per le bag patient-level

In [4]:
patient_to_rows = {
    patient_id:
        turn_dn4.loc[
            turn_dn4["patient_id"]
            == patient_id,
            "embedding_row"
        ]
        .to_numpy(dtype=int)

    for patient_id in patient_ids
}

patient_to_label = dict(
    zip(
        patient_ids,
        patient_labels
    )
)

print(
    "Bag create:",
    len(patient_to_rows)
)

bag_sizes = np.array(
    [
        len(patient_to_rows[p])
        for p in patient_ids
    ]
)

print(
    "Turni/bag | min:",
    bag_sizes.min(),
    "| mediana:",
    np.median(bag_sizes),
    "| max:",
    bag_sizes.max()
)

Bag create: 90
Turni/bag | min: 8 | mediana: 39.0 | max: 97


21.4 — Training MIL per fold

In [5]:
def train_mil_fold(
    X_full,
    train_patients,
    test_patients,
    fold_seed,
    epochs=60
):

    torch.manual_seed(
        fold_seed
    )

    np.random.seed(
        fold_seed
    )

    # PCA SOLO sui turni dei pazienti training
    train_rows = np.concatenate(
        [
            patient_to_rows[p]
            for p in train_patients
        ]
    )

    pca = PCA(
        n_components=0.95,
        svd_solver="full"
    )

    pca.fit(
        X_full[
            train_rows
        ]
    )

    X_pca = pca.transform(
        X_full
    ).astype(
        np.float32
    )

    model = AttentionMIL(
        input_dim=X_pca.shape[1],
        hidden_dim=64,
        attention_dim=32,
        dropout=0.20
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    criterion = (
        nn.BCEWithLogitsLoss()
    )

    rng = np.random.default_rng(
        fold_seed
    )

    # Training
    for epoch in range(
        epochs
    ):
        model.train()
        shuffled_patients = (
            rng.permutation(
                train_patients
            )
        )

        for patient_id in shuffled_patients:
            rows = patient_to_rows[
                patient_id
            ]

            bag = torch.tensor(
                X_pca[rows],
                dtype=torch.float32,
                device=DEVICE
            )

            label = torch.tensor(
                float(
                    patient_to_label[
                        patient_id
                    ]
                ),
                dtype=torch.float32,
                device=DEVICE
            )

            optimizer.zero_grad()

            logit, _ = model(
                bag
            )

            loss = criterion(
                logit,
                label
            )

            loss.backward()

            optimizer.step()

    # Test patient-level
    model.eval()
    prediction_rows = []

    with torch.inference_mode():

        for patient_id in test_patients:
            rows = patient_to_rows[
                patient_id
            ]

            bag = torch.tensor(
                X_pca[rows],
                dtype=torch.float32,
                device=DEVICE
            )

            logit, attention = model(
                bag
            )

            probability = torch.sigmoid(
                logit
            ).item()

            prediction_rows.append(
                {
                    "patient_id":
                        patient_id,

                    "y_true":
                        patient_to_label[
                            patient_id
                        ],

                    "probability":
                        probability,

                    "y_pred":
                        int(
                            probability
                            >= 0.5
                        ),

                    "n_turns":
                        len(rows)
                }
            )

    predictions = pd.DataFrame(
        prediction_rows
    )

    return (
        predictions,
        pca.n_components_
    )

21.5 — 5-fold MIL Wav2Vec2

In [6]:
import time

MIL_SEEDS = [
    42,
    123,
    456
]

mil_fold_rows = []
mil_oof_rows = []

start_total = time.time()

for fold_number, (
    train_patients,
    test_patients
) in enumerate(
    patient_folds,
    start=1
):

    print(
        f"\n===== FOLD {fold_number} ====="
    )

    seed_predictions = []

    for seed in MIL_SEEDS:
        predictions, n_components = (
            train_mil_fold(
                X_wav2vec2,
                train_patients,
                test_patients,
                fold_seed=seed
                + fold_number,
                epochs=60
            )
        )

        seed_predictions.append(
            predictions[
                [
                    "patient_id",
                    "probability"
                ]
            ]
            .rename(
                columns={
                    "probability":
                        f"probability_{seed}"
                }
            )
        )

    # Ensemble dei 3 seed
    ensemble = (
        seed_predictions[0]
        .merge(
            seed_predictions[1],
            on="patient_id"
        )
        .merge(
            seed_predictions[2],
            on="patient_id"
        )
    )

    probability_cols = [
        c
        for c in ensemble.columns
        if c.startswith(
            "probability_"
        )
    ]

    ensemble[
        "probability"
    ] = (
        ensemble[
            probability_cols
        ]
        .mean(axis=1)
    )

    ensemble[
        "y_true"
    ] = (
        ensemble[
            "patient_id"
        ]
        .map(
            patient_to_label
        )
        .astype(int)
    )

    ensemble[
        "y_pred"
    ] = (
        ensemble[
            "probability"
        ] >= 0.5
    ).astype(int)

    metrics = {
        "accuracy":
            accuracy_score(
                ensemble["y_true"],
                ensemble["y_pred"]
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                ensemble["y_true"],
                ensemble["y_pred"]
            ),

        "f1":
            f1_score(
                ensemble["y_true"],
                ensemble["y_pred"],
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                ensemble["y_true"],
                ensemble["probability"]
            )
    }

    mil_fold_rows.append(
        {
            "fold":
                fold_number,

            "n_components":
                n_components,

            **metrics
        }
    )

    ensemble[
        "fold"
    ] = fold_number


    mil_oof_rows.append(
        ensemble
    )

    print(
        f"features={n_components} | "
        f"BA={metrics['balanced_accuracy']:.4f} | "
        f"AUC={metrics['roc_auc']:.4f} | "
        f"F1={metrics['f1']:.4f}"
    )

mil_results_df = pd.DataFrame(
    mil_fold_rows
)

mil_oof_df = pd.concat(
    mil_oof_rows,
    ignore_index=True
)

print(
    "\nTempo totale:",
    round(
        (
            time.time()
            - start_total
        ) / 60,
        1
    ),
    "minuti"
)


===== FOLD 1 =====
features=115 | BA=0.5000 | AUC=0.5062 | F1=0.4706

===== FOLD 2 =====
features=115 | BA=0.5556 | AUC=0.4938 | F1=0.6364

===== FOLD 3 =====
features=116 | BA=0.4444 | AUC=0.5185 | F1=0.4444

===== FOLD 4 =====
features=115 | BA=0.5556 | AUC=0.6543 | F1=0.4286

===== FOLD 5 =====
features=116 | BA=0.4875 | AUC=0.4000 | F1=0.5714

Tempo totale: 1.9 minuti


21.6 — Riepilogo MIL

In [7]:
print(
    "===== MEDIA 5 FOLD ====="
)

display(
    mil_results_df[
        [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "roc_auc"
        ]
    ]
    .agg(
        ["mean", "std"]
    )
    .round(4)
)

assert (
    len(mil_oof_df) == 90
)

assert (
    mil_oof_df[
        "patient_id"
    ].nunique() == 90
)

mil_oof_metrics = {
    "accuracy":
        accuracy_score(
            mil_oof_df["y_true"],
            mil_oof_df["y_pred"]
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            mil_oof_df["y_true"],
            mil_oof_df["y_pred"]
        ),

    "f1":
        f1_score(
            mil_oof_df["y_true"],
            mil_oof_df["y_pred"],
            zero_division=0
        ),

    "roc_auc":
        roc_auc_score(
            mil_oof_df["y_true"],
            mil_oof_df["probability"]
        )
}

print(
    "\n===== OOF 90 PAZIENTI ====="
)

for metric, value in (
    mil_oof_metrics.items()
):

    print(
        f"{metric:20s}: "
        f"{value:.4f}"
    )

===== MEDIA 5 FOLD =====


,accuracy,balanced_accuracy,f1,roc_auc
mean,0.5111,0.5086,0.5103,0.5146
std,0.0465,0.0476,0.0898,0.0911



===== OOF 90 PAZIENTI =====
accuracy            : 0.5111
balanced_accuracy   : 0.5109
f1                  : 0.5217
roc_auc             : 0.5262


21.7 — Salvataggio 

In [ ]:
mil_results_df.to_csv(
    MIL_DIR / "risultati_5fold_MIL_Wav2Vec2_DN4.csv",
    index=False
)

mil_oof_df.to_csv(
    MIL_DIR / "predizioni_OOF_MIL_Wav2Vec2_DN4.csv",
    index=False
)

mil_oof_summary_df = pd.DataFrame(
    [
        {
            "representation": "wav2vec2",
            "model": "Attention_MIL",
            **mil_oof_metrics
        }
    ]
)

mil_oof_summary_df.to_csv(
    MIL_DIR / "riepilogo_OOF_MIL_Wav2Vec2_DN4.csv",
    index=False
)

print("✓ Notebook 21 salvato.")
print(MIL_DIR)

display(
    mil_oof_summary_df.round(4)
)

✓ Notebook 21 salvato.
C:\Users\acer\Desktop\ProgettoTesi\risultati\mil_wav2vec2_dn4


,representation,model,accuracy,balanced_accuracy,f1,roc_auc
0,wav2vec2,Attention_MIL,0.5111,0.5109,0.5217,0.5262
